##### [실전] 타이타닉 생존률 예측 모형을 위한 데이터 전 처리

In [ ]:
import pandas as pd
titanic_url='https://github.com/sehakflower/data/blob/main/titanic_1309.xlsx?raw=true'
titanic=pd.read_excel(titanic_url,sheet_name='total')
train_1000=titanic.iloc[:1000]
test_309=titanic.iloc[1000:]
train_1000.drop(['boat', 'body', 'home.dest'], axis=1, inplace=True)
test_309.drop(['boat', 'body', 'home.dest'], axis=1, inplace=True)
train_df=train_1000
test_df=test_309
train_df

In [ ]:
train_df.info( )

In [ ]:
total=[train_df, test_df] # 학습 및 테스트 데이터 세트 결합
for dataset in total:
    #""" title 찾기.  example) "Braund, Mr. Owen Harris" ->  Mr """
  dataset['title']=dataset['name'].str.extract('([A-za-z]+)\.', expand=False)       # expand: DataFrame 이 아니라 pd.Series 로 가져 오라는 옵션 
  
train_df['title'].unique( )

In [ ]:
title_mapping={"Mr":1, "Miss":2, 'Ms':2, 'Mlle':2, "Mrs":3, 'Mme':3, 'Master':4, 'Dr':5, 'Rev':5, 'Col': 5, 'Major':5, 'Lady':5, 'Capt':5, 'Sir':5, 'Don':5,
'Jonkheer':5, 'Countess':5}
for dataset in total:
  dataset['title']=dataset['title'].map(title_mapping)
  
train_df['title'].unique( )

In [ ]:
# 5개 그룹으로 만들기
total=[train_df, test_df]
titles={'Mr':1, 'Miss':2, 'Mrs':3, 'Master':4, 'Special':5}
for dataset in total:
  dataset['title']=dataset.name.str.extract('([A-Za-z]+)\.', expand=False)
  dataset['title']=dataset['title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Special')
  dataset['title']=dataset['title'].replace('Mlle', 'Miss')
  dataset['title']=dataset['title'].replace('Ms', 'Miss')
  dataset['title']=dataset['title'].replace('Mme', 'Mrs')
  dataset['title']=dataset['title'].map(titles) # 타이틀을 숫자로 변경
  dataset['title']=dataset['title'].fillna(0) # NaN을 0으로 변경
  
train_df=train_df.drop(['name'], axis=1)
test_df=test_df.drop(['name'], axis=1)
test_df

In [ ]:
#### age 열 처리
## 각 title (Mr, Mrs, Master, ...) 중에서 NaN 인 항목이 있으면 전체 median 값 1개를 가지고 채워 넣는것이 아니라 각 title group 의 평균 값으로 채워 넣음

# 최신 권장 방식
train_df['age'] = train_df['age'].fillna(train_df.groupby('title')['age'].transform('median'))
test_df['age'] = test_df['age'].fillna(test_df.groupby('title')['age'].transform('median'))

## 옛날 방식
##train_df['age'].fillna(train_df.groupby('title')['age'].transform('median'), inplace=True)
##test_df['age'].fillna(test_df.groupby('title')['age'].transform('median'), inplace=True)


test_df['age']

In [ ]:
## 나이 분포 가 어떻게 되고 각 나이 분포별로  survived 가 어떻게 되는지..
import matplotlib.pyplot as plt
import seaborn as sns

### FacetGrid graph 는 곡선 아래쪽의 면적이 1이 되게 만듬. 그러한 조건에서의 density 값.

## FacetGrid: grid 를 만드는 객체.  hue: 생존자와 사망자를 색으로 다르게,  aspect:  가로 / 세로 비율
facet=sns.FacetGrid(train_1000, hue="survived", aspect=4)

## kdeplot: Kernel Density Estimation.  histogram 을 부드러운 곡선으로 연결한 형태. fill:  곡선 아래 부분을 채움
#facet.map(sns.kdeplot, 'age', shade=True)  <- old version
facet.map(sns.kdeplot, 'age', fill=True)    

## x축의 나이를 0 세 부터 max 나이로 함
facet.set(xlim=(0, train_1000['age'].max( )))

# 그래프 오른쪽에 범례 (여기서는 survived 색이 구분되는 값)
facet.add_legend( )

sns.axes_style("darkgrid")
plt.show( )

In [ ]:
# 동일한 결과를 내는 최신 스타일 코드
sns.displot(data=train_1000, x="age", hue="survived", kind="kde", fill=True, aspect=4)
plt.xlim(0, train_1000['age'].max())
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 스타일 설정
sns.set_theme(style="darkgrid")

# displot을 이용한 통합 시각화
g = sns.displot(
    data=train_1000, 
    x="age", 
    hue="survived", 
    col="sex",      # 성별에 따라 그래프를 옆으로 나란히 배치 (3차원 확장)
    kind="kde",     # 곡선 밀도 그래프 형태 선택
    fill=True,      # 곡선 아래 면적 채우기
    aspect=1.5,     # 각 그래프의 가로 세로 비율
    height=4,       # 그래프의 높이 설정
    palette="husl"  # 색상 테마 설정
)

# x축 범위 제한 (0세부터 최대 나이까지)
g.set(xlim=(0, train_1000['age'].max()))

plt.show()

In [ ]:
data=[train_df, test_df]
for dataset1 in data:
  dataset1['age']=dataset1['age'].astype(int)
  dataset1.loc[ dataset1['age']<=11, 'age']=0
  dataset1.loc[(dataset1['age']>11)&(dataset1['age']<=18), 'age']=1
  dataset1.loc[(dataset1['age']>18)&(dataset1['age']<=22), 'age']=2
  dataset1.loc[(dataset1['age']>22)&(dataset1['age']<=27), 'age']=3
  dataset1.loc[(dataset1['age']>27)&(dataset1['age']<=33), 'age']=4
  dataset1.loc[(dataset1['age']>33)&(dataset1['age']<=40), 'age']=5
  dataset1.loc[(dataset1['age']>40)&(dataset1['age']<=66), 'age']=6
  dataset1.loc[ dataset1['age']>66, 'age']=6
  
train_df['age'].value_counts( )

In [ ]:
sex_mapping={'male':0, 'female':1}
for dataset in data:
  dataset['sex']=dataset['sex'].map(sex_mapping)
  
train_df

In [ ]:
## pclass
## embarked: 승선 항구.   C: Cherbourg (쉘부르 - 프랑스), Q: Queenstown (퀸즈타운- 아알랜드), S: Southampton (사우샘프턴 - 영국)
## 목적지는 미국 뉴욕

pclass1=train_df[train_df['pclass']==1]['embarked'].value_counts( )
pclass2=train_df[train_df['pclass']==2]['embarked'].value_counts( )
pclass3=train_df[train_df['pclass']==3]['embarked'].value_counts( )
df=pd.DataFrame([pclass1, pclass2, pclass3])

df.index=['1st class', '2nd class', '3rd class']
df.plot(kind='bar', stacked=True, figsize=(10, 5))

In [ ]:
# 각 항구별 탑승객 수 확인
print(train_df['embarked'].value_counts())

# 항구별 생존율 비교 (평균값)
print(train_df.groupby('embarked')['survived'].mean())

In [ ]:
## embarked: 승객이 배에 올라탄 항구.
## S: Southampton (영국: 출발지),  C: Cherbourg(프랑스: 부유한 승객 많음.),  Q: Queenstown(아일랜드:  미국으로 이민 가려는 3등석이 많음)
for dataset in data:
  dataset['embarked']=dataset['embarked'].fillna('S')
len(train_df['embarked'].isnull( ))

In [ ]:
embarked_mapping={'S':0, 'C':1, 'Q':2}
for dataset in data: # data=[train_df, test_df]
  dataset['embarked']=dataset['embarked'].map(embarked_mapping)
train_df['embarked']

In [ ]:
#### sibsp, parch

data=[train_df, test_df]
for dataset in data:
  dataset['sibpar']=dataset['sibsp']+dataset['parch']
  dataset.loc[dataset['sibpar']>0, 'n_alone']=0
  dataset.loc[dataset['sibpar']==0, 'n_alone']=1
  dataset['n_alone']=dataset['n_alone'].astype(int)
test_df

In [ ]:
train_df['n_alone'].value_counts( )

test_df['n_alone'].value_counts( )

In [ ]:
## cabin
## cabin 은 객실 번호를 뜻한다.
## A, B, C 데크:  상층부.  주로 1등석 승객이 머뭄
## D,E:  중간부.  주로 1~2 등석 승객이 머뭄
## F, G:  저층부(엔진근처) 주로 3등석 승객이 머뭄 

### 갑판의 높이가 구명보트까지 가는 시간과 밀접한 연관이 있슴.  하층부는 복잡한 미로를 빠져 나와서 상층부 갑판으로 가야 했슴

### cabin 이 "C22 C26"  이렇게 여러개 있을 수 있는 이유는 그 당시 한명이 여러 사람의 ticket 을 사면 ticket 번호가 동일 했슴


train_df.cabin.value_counts( )

#train_df.head(10)

In [ ]:
for dataset in data:
  dataset['cabin']=dataset['cabin'].str[:1]
train_df.cabin.value_counts()

In [ ]:
######
# A 가 적은 이유:  A 객실은 가장 좋아서 갯수 자체가 많지 않음
# C,B 가 많은 이유:  1등석 객실이 가장 많이 위치
# D 가 많은 이유:  식당 근처라서
# E:  1/2/3 등석이 골고루 많이 배치 됨

pclass1=train_df[train_df['pclass']==1]['cabin'].value_counts( )
pclass2=train_df[train_df['pclass']==2]['cabin'].value_counts( )
pclass3=train_df[train_df['pclass']==3]['cabin'].value_counts( )
df=pd.DataFrame([pclass1, pclass2, pclass3])

df.index=['1st class', '2nd class', '3rd class']
df.plot(kind='bar', stacked=True, figsize=(10, 5))

In [ ]:
#train_df["fare"].fillna(train_df.groupby('pclass')['fare'].transform('median'), inplace=True)
#test_df["fare"].fillna(test_df.groupby('pclass')['fare'].transform('median'), inplace=True)

# train_df["fare"] = train_df["fare"].fillna(train_df.groupby('pclass')['fare'].transform('median'))
# test_df["fare"] = test_df["fare"].fillna(test_df.groupby('pclass')['fare'].transform('median'))


# for dataset in data:
#   dataset.loc[ dataset['fare']<=20, 'fare']=1
#   dataset.loc[(dataset['fare']>20)&(dataset['fare']<=30), 'fare']=2
#   dataset.loc[(dataset['fare']>30)&(dataset['fare']<=50), 'fare']=3
#   dataset.loc[(dataset['fare']>50)&(dataset['fare']<=100), 'fare']=4
#   dataset.loc[ dataset['fare']>100, 'fare']=5
# train_df

In [ ]:
#### fare  열  (표 값)
import pandas as pd
import numpy as np

# 데이터프레임 리스트 (train_df, test_df가 정의되어 있다고 가정)
data = [train_df, test_df]

for dataset in data:
    # 1. Fare 결측치 처리 (직접 할당 방식 - 워닝 방지 핵심)
    # pclass별 중앙값으로 채우기
    dataset['fare'] = dataset['fare'].fillna(
        dataset.groupby('pclass')['fare'].transform('median')
    )
    
    # 2. Fare 범주화 (Categorization)
    # .loc을 사용할 때도 원본 객체에 직접 반영되도록 설계
    dataset.loc[ dataset['fare'] <= 20, 'fare'] = 1
    dataset.loc[(dataset['fare'] > 20) & (dataset['fare'] <= 30), 'fare'] = 2
    dataset.loc[(dataset['fare'] > 30) & (dataset['fare'] <= 50), 'fare'] = 3
    dataset.loc[(dataset['fare'] > 50) & (dataset['fare'] <= 100), 'fare'] = 4
    dataset.loc[ dataset['fare'] > 100, 'fare'] = 5
    
    # 데이터 타입을 정수형으로 명시 (선택 사항, 모델 학습에 유리)
    dataset['fare'] = dataset['fare'].astype(int)

# 결과 확인
train_df

In [ ]:
## 개인별 요금
for dataset1 in data:
  dataset1['fare_person']=dataset1['fare']/(dataset1['sibpar']+1)
  dataset1['fare_person']=dataset1['fare_person'].astype(int)
train_df

In [ ]:
## 최종 train data 
##

X_columns=['pclass', 'sex', 'age', 'embarked', 'title', 'sibpar', 'n_alone', 'fare_person']
y_column=['survived']
X_train=train_df[X_columns]
y_train=train_df[y_column]
y_train
X_train

In [ ]:
## 최종 test data 
##

X_columns=['pclass', 'sex', 'age', 'embarked', 'title', 'sibpar', 'n_alone', 'fare_person']
y_column=['survived']
X_test=test_df[X_columns]
y_test=test_df[y_column]
X_test
y_test